In [16]:
import os
os.environ["IANTIRTA_API_KEY"] = "e34abc2e2f51c02a056a1e5ff1d6f26e749695df"

In [ ]:
#@title Karaoke Plus
from __future__ import annotations

import os
from pathlib import Path
import subprocess
import typing as t
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, field
import concurrent.futures
import requests
import argparse
import logging

import kplus
from kplus import env, config
from kplus.pipelines.separate import BaseSeparator, DemucsSeparator
from kplus.pipelines.utils import ASRResult
from kplus.pipelines import (
    ensure_file,
    transcribe,
    align,
    align2ref,
    detect_audio_activity,
)
from kplus.tools.render import Render


# Logging Setup
opt = argparse.Namespace(log_level="debug")
config.parse_config(opt, setup_logging=True)
logger = logging.getLogger(__name__)

# Secret getter
if kplus.env.is_kaggle:
    from kaggle_secrets import UserSecretsClient
    get_secret = UserSecretsClient().get_secret
elif kplus.env.is_colab:
    from google.colab import userdata
    get_secret = userdata.get
else:
    get_secret = os.environ.get


TaskStatus: t.TypeAlias = t.Literal["waiting", "processing", "completed", "error"]
TaskType: t.TypeAlias = t.Literal["basic", "plus"]

@dataclass(slots=True)
class Task:
    """ Hold Task Type """
    id: int
    title: str
    artist: str
    duration: float
    lyrics: str
    status: TaskStatus
    url: str
    karaoke_type: TaskType

    error: str | None = field(init=False, default=...)
    videopath: str | None = field(init=False, default=...)
    karaokepath: str | None = field(init=False, default=...)

    @classmethod
    def from_dict(cls, data: dict) -> Task:
        return cls(
            id=data["id"],
            title=data["title"],
            artist=data["artist"],
            duration=data["duration"],
            lyrics=data["lyrics"],
            status=data["status"],
            url=data["url"],
            karaoke_type=data["karaoke_type"],
        )


tasks: list[Task] = []


class APIError(Exception):
    """ Handle API Error """


class IantirtaAPI:
    """ API Needed for connection to iantirta.com. """
    def __init__(self):
        self.api_token = get_secret("IANTIRTA_API_KEY")

    def post(self, url: str, json: dict):
        headers = {
            "Authorization": f"bearer {self.api_token}",
        }
        try:
            res = requests.post(url, headers=headers, json=json)
            res.raise_for_status()
            res = res.json()
            if res.get("error", False):
                raise APIError("API Error:", res)
            else:
                return res["result"]
        except Exception:
            raise

    def get_task(self, status_to_fetch: TaskStatus) -> list[Task]:
        api_url = "http://localhost:8069/karaoke/task"
        json = {"params": {
            "status_to_fetch": status_to_fetch,
        },}
        res = self.post(api_url, json)

        global tasks
        tasks = [Task.from_dict(r) for r in res]
        return tasks

class DriveAPI:
    """ API Needed for connection to google.drive. """
    SCOPES = ['https://www.googleapis.com/auth/drive']  # noqa: RUF012
    

class CloudflareAPI:
    """ API Needed for connection to cloudflare storage. """

class KaraokeWorker:
    def __init__(self):
        self.cookiefile = "cookies.txt"

    def _run_render(
        self,
        task: Task,
        instrument_path: str,
        result: ASRResult | None = None
    ) -> None:
        task.karaokepath = Render(with_ass=task.karaoke_type == 'plus').render(
            video_filepath=task.videopath,
            inst_path=instrument_path,
            duration=task.duration,
            result=result,
            output_path=None,
        )
        task.status = "completed"

    def _run(
        self,
        task: Task,
        separator: DemucsSeparator,
        executor: ThreadPoolExecutor,
    ) -> None:
        # Normalize to wav
        inputpath = Path(task.videopath).expanduser().resolve()
        mixpath = inputpath.stem + ".wav"
        subprocess.run([
            "ffmpeg", "-y", "-i",
            str(inputpath), "-vn",
            "-ar", str(separator.sr),
            "-ac", str(separator.ac),
            str(mixpath)
        ], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

        # Run separation
        sep_result = separator.separate(mixpath)

        if os.path.exists(mixpath):
            os.unlink(mixpath)
        
        if task.karaoke_type == "basic":
            # Imidiately pass it to render & upload
            instrument_path = sep_result.inst_path
            executor.submit(
                self._run_render,
                task,
                instrument_path,
            )
        elif task.karaoke_type == "plus":
            # Pass here since it needs to be run separately.
            # Run Detection & Transcribe
            try:
                audio_result = detect_audio_activity(
                    audio=sep_result.vocs_path,
                    sr=sep_result.sr,
                    no_show=True,
                    resample=False,
                    use_html=False,
                )
                asr_result = transcribe(
                    audio=sep_result.vocs_path,
                    audiosegments=audio_result.segments,
                    reference=task.lyrics,
                    languages=None,
                    transcribe_return_timestamps=True,
                    transcribe_model_name_or_path="Qwen/Qwen3-ASR-1.7B-hf",
                    max_inference_batch_size=1,
                    num_beams=4,
                )
                ref_result, new_audiosegments = align2ref(
                    asr_result,
                    task.lyrics,
                    audio_result.segments,
                    raise_if_not_reliable=False
                )
                align_results = align(sep_result.vocs_path, ref_result, new_audiosegments)
                align_results.to_line_idx(task.lyrics)
                result = align_results.populate_ass()
            except Exception:
                raise
        else:
            raise RuntimeError(
                f"Karaoke Type `{task.karaoke_type}` "
                "isn't supported yet."
            )
    
    def run(self) -> None:
        """ Pipeline
            * basic_tasks: (download, separate, render)
            * plust_tasks: (download, separate, transcribe, render)
        """
        render_executor = ThreadPoolExecutor(max_workers=3)
        separator = BaseSeparator.from_options(
            demucs="mdx_extra_q",
            overlap=0.75,
            segment=200,
            shifts=1,
            num_workers=0,
        )

        with ThreadPoolExecutor(num_workers=5) as executor:
            iter_task = iter(tasks)
            active_futures: dict[t.Any, Task] = {}
            for _ in range(5):
                if task := next(iter_task, None):
                    future = executor.submit(
                        ensure_file,
                        inputpath=task.url,
                        external_id=task.id,
                        no_lyrics=task.karaoke_type == "basic",
                        cookiefile=self.cookiefile,
                    )
                    active_futures[future] = task
            while active_futures:
                done, _ = concurrent.futures.wait(
                    active_futures.keys(),
                    return_when=concurrent.futures.FIRST_COMPLETED
                )
                for future in done:
                    task = active_futures.pop(future)
                    if next_task := next(iter_task, None):
                        new_future = executor.submit(
                            ensure_file,
                            inputpath=next_task.url,
                            external_id=next_task.id,
                            no_lyrics=next_task.karaoke_type == "basic",
                            cookiefile=self.cookiefile,
                        )
                        active_futures[new_future] = next_task
                    try:
                        info = future.result()
                        assert info.duration == task.duration, (
                            "Duration Missmatch between "
                            "server side and worker side, "
                            f"{info.duration} | {task.duration}"
                        )
                        assert info.title == task.title, (
                            "Title Missmatch between "
                            "server side and worker side, "
                            f"{info.title} | {task.title}"
                        )
                        assert info.artist == task.artist, (
                            "Artist Missmatch between "
                            "server side and worker side, "
                            f"{info.artist} | {task.artist}"
                        )
                        task.videopath = info.filepath
                        self._run(task, separator=separator, executor=render_executor)
                    except Exception as e:
                        logger.error(f"Worker Failed for Task {task.id}: {e}", exc_info=True)
                        task.update_status("failed", str(e))

        del separator.model, separator
        env.clean()

        for task in tasks:
            if task.karaoke_type == "basic":
                continue


        render_executor.shutdown()
        
        logger.info("Complete: All tasks processed through pipeline.")
        logger.info("complete")


from pprint import pprint
task = IantirtaAPI().get_task("waiting")
pprint(task)
pprint(tasks)

[Task(id=15,
      title='Bernadya - Kini Mereka Tahu (Official Video)',
      artist='Bernadya',
      duration=286.0,
      lyrics='Dari dulu kulebih-lebihkan semua\n'
             'Padahal yang kaulakukan tak seberapa\n'
             'Agar seisi dunia tahu\n'
             'Dan anggapku paling beruntung\n'
             'Milikimu\n'
             '\n'
             'Kukarang cerita yang semula tak ada\n'
             'Caraku sampaikan seolah semua nyata\n'
             'Agar semuanya setuju\n'
             'Dan yakin pada pilihanku\n'
             'Memilihmu\n'
             '\n'
             'Sifat baikmu yang orang tahu\n'
             'Itu karanganku\n'
             'Sifat aslimu yang hancurkanku\n'
             'Mereka tak tahu\n'
             '\n'
             'Dan bahkan setelah semua\n'
             'Yang kaulakukan padaku\n'
             'Ku tetap bela kamu\n'
             'Di depan teman-temanku\n'
             '\n'
             'Dan mungkin saja bisa jadi\n'
             'Bila 